# 04 — Minimum Spanning Trees (Kruskal and Prim)

## Why This Matters for DSA
A **Spanning Tree** is a connected, acyclic subgraph that spans (connects) all vertices of an undirected graph. A **Minimum Spanning Tree (MST)** is a spanning tree whose sum of edge weights is minimized.

MSTs represent a classic family of resource optimization problems: connecting houses to a water main with the least piping, establishing a power grid between cities with the least cabling, or designing a computer network connecting several routers. MSTs also serve as building blocks for clustering algorithms (Single-Linkage Clustering) and approximation algorithms for NP-hard routing problems.

In this notebook, we explore the two classic greedy MST algorithms: **Kruskal's** (edge-centric) and **Prim's** (node-centric).

## Prerequisites
- `01_Representations_DFS_BFS` (Phase 04) — Adjacency lists and matrix graphs.
- `02_Topological_Sort_and_DSU` (Phase 04) — Disjoint Set Union (DSU) core operations.
- `01_Heaps_and_Priority_Queues` (Phase 03) — Priority queue operations.

## Learning Objectives
By the end of this notebook, you will be able to:
- Explain what a Spanning Tree and a Minimum Spanning Tree (MST) are.
- Implement Kruskal's Algorithm using edge sorting and Disjoint Set Union (DSU).
- Implement Prim's Algorithm using a min-priority queue.
- Choose the correct MST algorithm based on graph density (sparse vs. dense).
- Verify the performance profiles of Kruskal's vs. Prim's algorithms empirically.

## Checklist Before Moving On
- [ ] I can state the definition of a Spanning Tree and explain why it must have exactly $V-1$ edges.
- [ ] I can write Kruskal's algorithm from memory using sorting and DSU.
- [ ] I can explain why sorting dominates the time complexity of Kruskal's.
- [ ] I can write Prim's algorithm from memory using C++ `std::priority_queue`.
- [ ] I can select between Kruskal's and Prim's depending on edge count $E$ relative to vertex count $V$.

## Table of Contents
1. Spanning Tree Properties
2. Kruskal's Algorithm — Edge-Centric Greedy Approach
3. Prim's Algorithm — Node-Centric Greedy Approach
4. Benchmarking: Kruskal vs. Prim
5. Checkpoint, Mini Project, and Answer Key
6. LeetCode Practice Problems

## 1. Spanning Tree Properties

### The Why
Before implementing MST algorithms, we must understand their structural invariants. If we know these mathematical rules, we can optimize our code's termination checks, preventing redundant operations.

### Core Mechanism
For an undirected, connected graph $G = (V, E)$, a Spanning Tree has the following key properties:
1. **Connectivity:** It connects all $V$ vertices.
2. **Acyclic:** It contains no cycles.
3. **Edge Count:** It contains exactly **$V - 1$ edges**. 
   - *Proof:* A single node has 0 edges. Every new node added to the component requires exactly 1 edge to connect it without creating a cycle. Thus, $V$ nodes require exactly $V-1$ edges.
   - If we add even 1 more edge, we are guaranteed to create a cycle.
   - If we remove even 1 edge, the graph is guaranteed to become disconnected.

This $V-1$ edge invariant is incredibly useful. In both Kruskal's and Prim's algorithms, we can immediately terminate our loops once our MST edge count reaches $V-1$. This prevents wasteful scans over remaining edges in dense graphs.

```
       Original Graph:                     Spanning Tree (V=4, E=3):
         (0) --[2]-- (1)                        (0) --[2]-- (1)
          |           |                          |
         [6]         [3]                        [6]
          |           |                          |
         (3) --[9]-- (2)                        (3)         (2)
                                                (No cycle, connected)
```

## 2. Kruskal's Algorithm — Edge-Centric Greedy Approach

### The Why
Kruskal's algorithm is an edge-centric greedy algorithm. It builds the MST by growing multiple disconnected components and merging them step-by-step. It is highly intuitive: to build the cheapest tree, we should always greedily examine the cheapest remaining edge.

### Core Mechanism
1. **Edge List Representation:** Store all edges in a single flat list: `vector<Edge>`.
2. **Sort:** Sort the edge list in non-decreasing order of edge weights.
3. **DSU Initialization:** Initialize a DSU of size $V$ (representing $V$ isolated components).
4. **Greedy Merge:** Loop through the sorted edge list. For each edge $(u, v)$ with weight $w$:
   - Check if $u$ and $v$ are in different components using `find(u) != find(v)`.
   - If they are, they do not form a cycle. Add the edge to the MST, add its weight to the cost, and merge them: `unite(u, v)`.
   - If they are in the same component, ignore the edge (adding it would create a cycle).
   - **Early Termination:** Stop once the MST has exactly $V-1$ edges.

### Complexity
- **Time Complexity:** $O(E \log E)$ or $O(E \log V)$. Sorting $E$ edges takes $O(E \log E)$. DSU operations cost $O(E \cdot \alpha(V))$. Since $\log E \le \log(V^2) = 2 \log V$, the sorting cost dominates: $O(E \log V)$.
- **Space Complexity:** $O(V + E)$ to store edges and DSU parent arrays.

### Common Pitfalls
- **Incorrect DSU Unite Call:** Calling `unite(u, v)` *before* checking `find(u) != find(v)`. This will cause the DSU to merge already-connected elements and report incorrect cycle status.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <numeric>
#include <algorithm>

using namespace std;

class KruskalDSU {
    vector<int> parent;
    vector<int> size;

public:
    KruskalDSU(int n) {
        parent.resize(n);
        iota(parent.begin(), parent.end(), 0);
        size.assign(n, 1);
    }

    int find(int i) {
        if (parent[i] == i) return i;
        return parent[i] = find(parent[i]);
    }

    bool unite(int i, int j) {
        int rootI = find(i);
        int rootJ = find(j);
        if (rootI == rootJ) return false;

        if (size[rootI] < size[rootJ]) swap(rootI, rootJ);
        parent[rootJ] = rootI;
        size[rootI] += size[rootJ];
        return true;
    }
};

struct Edge {
    int u, v, weight;

    // Sort edges by weight
    bool operator<(const Edge& other) const {
        return weight < other.weight;
    }
};

class KruskalGraph {
    int V;
    vector<Edge> edges;

public:
    KruskalGraph(int vertices) : V(vertices) {}

    void addEdge(int u, int v, int weight) {
        edges.push_back({u, v, weight});
    }

    pair<int, vector<Edge>> getMST() {
        vector<Edge> sortedEdges = edges;
        sort(sortedEdges.begin(), sortedEdges.end());

        KruskalDSU dsu(V);
        int mstWeight = 0;
        vector<Edge> mstEdges;

        for (const auto& edge : sortedEdges) {
            if (dsu.unite(edge.u, edge.v)) {
                mstWeight += edge.weight;
                mstEdges.push_back(edge);
                if ((int)mstEdges.size() == V - 1) break; // MST complete
            }
        }
        return {mstWeight, mstEdges};
    }
};

int main() {
    cout << "--- Kruskal's Algorithm Demo ---\n";
    KruskalGraph g(5);
    g.addEdge(0, 1, 2);
    g.addEdge(0, 3, 6);
    g.addEdge(1, 2, 3);
    g.addEdge(1, 3, 8);
    g.addEdge(1, 4, 5);
    g.addEdge(2, 4, 7);
    g.addEdge(3, 4, 9);

    auto [cost, mst] = g.getMST();
    cout << "Total MST Cost: " << cost << "\nEdges in MST:\n";
    for (const auto& edge : mst) {
        cout << edge.u << " <-> " << edge.v << " (weight: " << edge.weight << ")\n";
    }

    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 3. Prim's Algorithm — Node-Centric Greedy Approach

### The Why
While Kruskal's is edge-centric, Prim's algorithm is **node-centric**. It starts from a single vertex and grows the MST outward by greedily choosing the cheapest edge that connects a node *inside* the tree to a node *outside* the tree.

Prim's is highly efficient for dense graphs because it does not require a full edge sort, relying instead on a min-priority queue to find the cheapest neighbor.

### Core Mechanism
1. **Initialize:** Mark all vertices as unvisited.
2. **Min-Priority Queue:** Use a min-heap to store pairs of `{weight, vertex}`. Push `{0, start}`.
3. **Iteration:** While the queue is not empty:
   - Extract the pair `{w, u}` with the minimum weight.
   - If $u$ is already in the MST (visited), skip it.
   - Mark $u$ as visited (add it to the MST), and add $w$ to the total cost.
   - For each neighbor $v$ of $u$ with edge weight $weight$:
     - If $v$ is not yet in the MST, push `{weight, v}` into the queue.

### Complexity
- **Time Complexity:** $O(E \log V)$ using an adjacency list and binary heap. Each edge is pushed/popped from the heap at most once.
- **Space Complexity:** $O(V + E)$ to store the adjacency list and priority queue.

### Common Pitfalls
- **Pop-Time vs Push-Time Visited Checking:** In Prim's, you **must not** mark a node as visited when pushing it into the queue. A node can have multiple candidate edges connecting it to the tree. We must allow these edges to be queued, and only mark the node as visited when we pop the *cheapest* edge.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <queue>

using namespace std;

class PrimGraph {
    int V;
    vector<vector<pair<int, int>>> adj; // pair of {neighbor, weight}

public:
    PrimGraph(int vertices) : V(vertices) {
        adj.resize(V);
    }

    void addEdge(int u, int v, int weight) {
        adj[u].push_back({v, weight});
        adj[v].push_back({u, weight}); // undirected
    }

    int getMST(int start = 0) {
        vector<bool> inMST(V, false);
        // Min-priority queue: stores pair of {weight, vertex}
        priority_queue<pair<int, int>, vector<pair<int, int>>, greater<pair<int, int>>> pq;

        int mstWeight = 0;
        pq.push({0, start}); // Seed with start node

        while (!pq.empty()) {
            auto [w, u] = pq.top();
            pq.pop();

            if (inMST[u]) continue;

            inMST[u] = true;
            mstWeight += w;

            for (const auto& edge : adj[u]) {
                int v = edge.first;
                int weight = edge.second;
                if (!inMST[v]) {
                    pq.push({weight, v});
                }
            }
        }
        return mstWeight;
    }
};

int main() {
    cout << "--- Prim's Algorithm Demo ---\n";
    PrimGraph g(5);
    g.addEdge(0, 1, 2);
    g.addEdge(0, 3, 6);
    g.addEdge(1, 2, 3);
    g.addEdge(1, 3, 8);
    g.addEdge(1, 4, 5);
    g.addEdge(2, 4, 7);
    g.addEdge(3, 4, 9);

    cout << "Total MST Cost: " << g.getMST(0) << "\n"; // Expected: 16

    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 4. Benchmarking: Kruskal vs. Prim

### The Why
To show the execution difference between Kruskal's sorting-dominated complexity and Prim's heap-dominated complexity, we benchmark them on a sparse graph of $V = 2000, E = 8000$.

Let's compile and run the benchmark.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <queue>
#include <numeric>
#include <algorithm>
#include <chrono>
#include <random>

using namespace std;

struct Timer {
    string name;
    chrono::high_resolution_clock::time_point start;
    Timer(const string& n) : name(n), start(chrono::high_resolution_clock::now()) {}
    ~Timer() {
        auto end = chrono::high_resolution_clock::now();
        auto diff = chrono::duration_cast<chrono::microseconds>(end - start).count();
        cout << name << ": " << diff << " microseconds\n";
    }
};

struct Edge {
    int u, v, weight;
    bool operator<(const Edge& other) const { return weight < other.weight; }
};

class DSU {
    vector<int> parent;
    vector<int> size;
public:
    DSU(int n) {
        parent.resize(n);
        iota(parent.begin(), parent.end(), 0);
        size.assign(n, 1);
    }
    int find(int i) {
        if (parent[i] == i) return i;
        return parent[i] = find(parent[i]);
    }
    bool unite(int i, int j) {
        int rootI = find(i);
        int rootJ = find(j);
        if (rootI == rootJ) return false;
        if (size[rootI] < size[rootJ]) swap(rootI, rootJ);
        parent[rootJ] = rootI;
        size[rootI] += size[rootJ];
        return true;
    }
};

int main() {
    const int V = 2000;
    const int E = 8000;

    mt19937 rng(42);
    uniform_int_distribution<int> distV(0, V - 1);
    uniform_int_distribution<int> distW(1, 1000);

    vector<Edge> edges;
    vector<vector<pair<int, int>>> adj(V);

    for (int i = 0; i < E; ++i) {
        int u = distV(rng);
        int v = distV(rng);
        int w = distW(rng);
        if (u != v) {
            edges.push_back({u, v, w});
            adj[u].push_back({v, w});
            adj[v].push_back({u, w});
        }
    }

    // Benchmark Kruskal
    {
        Timer t("Kruskal's Algorithm");
        vector<Edge> sortedEdges = edges;
        sort(sortedEdges.begin(), sortedEdges.end());
        DSU dsu(V);
        int mstWeight = 0;
        int count = 0;
        for (const auto& edge : sortedEdges) {
            if (dsu.unite(edge.u, edge.v)) {
                mstWeight += edge.weight;
                count++;
                if (count == V - 1) break;
            }
        }
        cout << "Kruskal MST Cost: " << mstWeight << "\n";
    }

    // Benchmark Prim
    {
        Timer t("Prim's Algorithm");
        vector<bool> inMST(V, false);
        priority_queue<pair<int, int>, vector<pair<int, int>>, greater<pair<int, int>>> pq;
        int mstWeight = 0;
        pq.push({0, 0});
        while (!pq.empty()) {
            auto [w, u] = pq.top();
            pq.pop();
            if (inMST[u]) continue;
            inMST[u] = true;
            mstWeight += w;
            for (const auto& edge : adj[u]) {
                int v = edge.first;
                int weight = edge.second;
                if (!inMST[v]) {
                    pq.push({weight, v});
                }
            }
        }
        cout << "Prim MST Cost: " << mstWeight << "\n";
    }

    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 5. Checkpoint, Mini Project, and Answer Key

### MST Algorithmic Decision Guide

| Algorithm | Complexity | Density Efficiency | Key Structure | Ideal Use Case |
|---|---|---|---|---|
| **Kruskal's** | $O(E \log V)$ | Excellent for Sparse | Edge List + DSU | Sparse graphs ($E \approx V$). Easy connection tracking. |
| **Prim's** | $O(E \log V)$ | Excellent for Dense | Adjacency List + Heap | Dense graphs ($E \approx V^2$). |

### Checkpoint Questions
1. Why does Kruskal's algorithm run faster on sparse graphs, while Prim's can be faster on dense graphs?
2. What happens if you run Kruskal's or Prim's algorithm on a graph that is disconnected?
3. Why does a spanning tree of a graph with $V$ vertices always have exactly $V-1$ edges?
4. In Prim's algorithm, why do we mark nodes as visited at pop-time, whereas in BFS we mark them at push-time?
5. How can we find the Maximum Spanning Tree of a graph using Kruskal's?
6. If all edge weights in a graph are identical, what is the cost of its Minimum Spanning Tree?

### Answer Key
1. Kruskal's requires sorting all edges, which is cheap when $E$ is small (sparse) but becomes expensive when $E \approx V^2$. Prim's only processes neighbors dynamically, avoiding sorting the entire edge set, making it highly efficient on dense graphs.
2. Kruskal's will complete and output a "Spanning Forest" containing disconnected components. Prim's will only span the component containing the starting vertex, leaving other components unvisited.
3. Every connected, acyclic graph (tree) of size $V$ has $V-1$ edges. Adding 1 edge creates a cycle; removing 1 edge disconnects the components.
4. BFS is unweighted, so the first path found is guaranteed to be shortest. Prim's is weighted: a node can be queued multiple times with different weights. We must allow these edges to reside in the queue and only commit to the cheapest one when we pop it.
5. Sort edges in non-increasing order of weight (descending) instead of ascending. DSU logic remains unchanged.
6. The cost is $(V - 1) \times W$, where $W$ is the constant edge weight.

### Mini Project: Fiber Cable Grid Optimizer
Design a layout of fiber-optic cables connecting several cities with the lowest total installation cost.
Write functions to:
1. Model cities and connection costs.
2. Build the optimal cabling layout using Kruskal's algorithm (MST).
3. List the connections that must be installed.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <string>
#include <unordered_map>
#include <numeric>
#include <algorithm>

using namespace std;

class FiberDSU {
    vector<int> parent;
    vector<int> size;

public:
    FiberDSU(int n) {
        parent.resize(n);
        iota(parent.begin(), parent.end(), 0);
        size.assign(n, 1);
    }

    int find(int i) {
        if (parent[i] == i) return i;
        return parent[i] = find(parent[i]);
    }

    bool unite(int i, int j) {
        int rootI = find(i);
        int rootJ = find(j);
        if (rootI == rootJ) return false;
        if (size[rootI] < size[rootJ]) swap(rootI, rootJ);
        parent[rootJ] = rootI;
        size[rootI] += size[rootJ];
        return true;
    }
};

struct Link {
    int u, v, cost;
    bool operator<(const Link& other) const { return cost < other.cost; }
};

class TelecomNetwork {
    int V;
    vector<Link> links;
    unordered_map<int, string> idToName;
    unordered_map<string, int> nameToId;

public:
    TelecomNetwork(const vector<string>& cities) {
        V = cities.size();
        for (int i = 0; i < V; ++i) {
            idToName[i] = cities[i];
            nameToId[cities[i]] = i;
        }
    }

    void addConnection(const string& from, const string& to, int cost) {
        int u = nameToId[from];
        int v = nameToId[to];
        links.push_back({u, v, cost});
    }

    pair<int, vector<Link>> buildOptimalGrid() {
        vector<Link> sortedLinks = links;
        sort(sortedLinks.begin(), sortedLinks.end());

        FiberDSU dsu(V);
        int totalCost = 0;
        vector<Link> mstLinks;

        for (const auto& link : sortedLinks) {
            if (dsu.unite(link.u, link.v)) {
                totalCost += link.cost;
                mstLinks.push_back(link);
                if ((int)mstLinks.size() == V - 1) break;
            }
        }
        return {totalCost, mstLinks};
    }

    string getCityName(int id) const {
        return idToName.at(id);
    }
};

int main() {
    vector<string> cities = {"London", "Paris", "Berlin", "Rome", "Madrid"};
    TelecomNetwork net(cities);

    net.addConnection("London", "Paris", 300);
    net.addConnection("London", "Berlin", 800);
    net.addConnection("Paris", "Berlin", 450);
    net.addConnection("Paris", "Rome", 700);
    net.addConnection("Berlin", "Rome", 500);
    net.addConnection("Madrid", "Rome", 900);
    net.addConnection("Madrid", "Paris", 600);

    cout << "--- Telecom Network Optimization ---\n";
    auto [cost, mst] = net.buildOptimalGrid();
    cout << "Minimum installation cost to connect all cities: $" << cost << "k\n";
    cout << "Connections to install:\n";
    for (const auto& link : mst) {
        cout << "  " << net.getCityName(link.u) << " <-> " << net.getCityName(link.v) << " (Cost: $" << link.cost << "k)\n";
    }

    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 6. LeetCode Practice Problems

| # | Problem | Difficulty | Topic | Hint |
|---|---|---|---|---|
| 1584 | Min Cost to Connect All Points | Medium | MST | Connect points on a 2D plane. Since the graph is fully connected (dense), Prim's algorithm runs extremely fast here. |
| 1168 | Optimize Water Distribution in a Village | Hard | MST Variant | Virtual node trick: create a virtual node representing the water supply. Add edges from this node to all houses with weight = cost of building a well. Then, run Kruskal's/Prim's. |
| 1489 | Find Critical and Pseudo-Critical Edges in MST | Hard | MST + Kruskal | Run Kruskal's to find base MST cost. To check if an edge is critical, delete it and see if MST cost increases. To check if pseudo-critical, force-include it and check if MST cost remains base. |
| 1135 | Connecting Cities With Minimum Cost | Medium | MST | Standard MST application on cities. If you cannot connect all cities (MST edges < $V-1$), return -1. |
| 1559 | Detect Cycles in 2D Grid | Medium | DFS / DSU | Run cycle detection on a grid graph where adjacent cells with the same value form edges. |

### Summary of Phase 04
Congratulations! You have completed **Phase 04 (Graph Theory)**:
1. **Representations & Traversal:** Adjacency List/Matrix, DFS connected components, Directed/Undirected cycle detection, and unweighted BFS shortest paths.
2. **Topological Sort & DSU:** Kahn's algorithm, DFS sorting, and Near-$O(1)$ DSU dynamic connectivity.
3. **Shortest Paths:** Dijkstra, Bellman-Ford (negative cycle checks), and Floyd-Warshall (APSP).
4. **Minimum Spanning Trees:** Kruskal's (DSU) and Prim's (Priority Queue) algorithms.

In the next phase, we transition to **Phase 05 (Algorithmic Paradigms)**, exploring recursion, divide-and-conquer, greedy algorithms, and dynamic programming.